In [ ]:
# import statements

import PyPhenom as ppi
import numpy as np
import matplotlib.pyplot as plt
import cv2

from math import factorial
from scipy.fft import fft2, fftshift
from scipy.signal import fftconvolve
from lmfit import Model, models, minimize, Minimizer, create_params
from skopt import gp_minimize

%matplotlib widget


In [ ]:
# connect to SEM 

PhenomID = 'MVE086101-20057-F'
username = 'MVE08610120057F'
password = 'KN68X6HJQMT9'

phenom = ppi.Phenom(PhenomID, username, password)

In [150]:
# global variables

def calibrate_kernel_size(type, val_min):
    """
    yea gotta change this

    Parameter wd_val_min: the minimum working distance (float) that the SEM can image at
    """

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    sums = -1
    kernel_size = 5
    ori_mag = ppi.MagnificationFromFieldWidth(phenom.GetHFW(), 0.52)

    if type == 'wd':
        phenom.SetHFW(ppi.MagnificationToFieldWidth(10000, 0.52))
        phenom.SetSemWD(val_min)

    elif type == "xst":
        phenom.SetHFW(ppi.MagnificationToFieldWidth(40000, 0.52))
        phenom.SetSemStigmate(ppi.Position(val_min, 0))

    elif type == "yst":
        phenom.SetHFW(ppi.MagnificationToFieldWidth(40000, 0.52))
        phenom.SetSemStigmate(ppi.Position(0, val_min))

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim, ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array

    while np.round(sums, 4) != 0:
        blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0)
        filtered_data = cv2.Canny(blurred_img, 50, 150) #50, 150, 9

        sums = filtered_data.sum()
        
        if np.round(sums, 4) != 0:
            kernel_size = kernel_size + 2

    phenom.SetHFW(ppi.MagnificationToFieldWidth(ori_mag, 0.52))
    phenom.SetSemStigmate(ppi.Position(0, 0))

    return kernel_size


wd_val_min = 0.002 #0.008 #0.002
wd_val_max = 0.012 #0.011 #0.025 #0.12198 is max value
xst_val_min = -0.2 #-1
xst_val_max = 0.2 #1
yst_val_min = -0.2 #-1
yst_val_max = 0.2 #1

size = calibrate_kernel_size('wd', wd_val_min)
x_size = calibrate_kernel_size('xst', xst_val_min)
y_size = calibrate_kernel_size('yst', yst_val_min)
print(size, x_size, y_size)



7 9 11


In [ ]:
# autocorrelation functions

def auto_correlation(frame):
    """
    Returns the computed auto_correlation for a given frame.

    Parameter frame: an SEM image
    """

    ft = np.fft.fft2(frame)
    autocorr = np.fft.ifft2(ft * np.conjugate(ft))
    return np.fft.fftshift(autocorr).real


def filtered_autocorr(img_array, kernel_size):
    """
    Returns the autocorrelation array for the given image array and the kernel size to blur the image.

    Parameter img_array: a Numpy array
    Parameter kernel_size: an int specifying the size of the blurring kernel
    """
    
    blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0) # blurs the image to help with noise
    filtered_data = cv2.Canny(blurred_img, 50, 150) # sets the values of edges = 255, everything else = 0
    acdata = auto_correlation(filtered_data) # calculates the auto_correlation data for the filtered image

    return acdata

    # blurred_img = cv2.GaussianBlur(data[wd, :, :], (9, 9), 0) 

    # orb = cv2.ORB_create()
    # kp = orb.detect(blurred_img, None)
    # kp, des = orb.compute(blurred_img, kp)

    # # img2 = cv.drawKeypoints(blank_image, kp, None, color=(0,255,0), flags=0)
    # # plt.imshow(img2), plt.show()

    # filtered[wd, :, :] = np.zeros(img.shape[:2], dtype=np.uint8)

    # for keypoint in kp:
    #     filtered[wd, int(keypoint.pt[0]), int(keypoint.pt[1])] = 255

    #     for row in range(-4, 4):
    #         for col in range(-4, 4):
    #             filtered[wd, int(keypoint.pt[0] + row), int(keypoint.pt[1] + col)] = 255

In [ ]:
# fitting functions

# Define basic Zernike polynomials (on the unit disk)
def zernike_radial(n, m, rho):
    R = np.zeros_like(rho)
    for k in range((n - abs(m)) // 2 + 1):
        coeff = (-1)**k * factorial(n - k)
        coeff /= (factorial(k) *
                  factorial((n + abs(m)) // 2 - k) *
                  factorial((n - abs(m)) // 2 - k))
        R += coeff * rho**(n - 2 * k)
    return R


def zernike(n, m, rho, theta):
    if m > 0:
        return zernike_radial(n, m, rho) * np.cos(m * theta)
    elif m < 0:
        return zernike_radial(n, -m, rho) * np.sin(-m * theta)
    else:
        return zernike_radial(n, 0, rho)


# Mapping Noll index to (n, m)
NOLL_MAP = {
    1:  (0,  0),   # Piston (constant phase offset)
    2:  (1, -1),   # Tilt X (Tip)
    3:  (1,  1),   # Tilt Y (Tilt)
    4:  (2, -2),   # Oblique Astigmatism
    5:  (2,  0),   # Defocus
    6:  (2,  2),   # Vertical Astigmatism
    7:  (3, -3),   # Vertical Trefoil
    8:  (3, -1),   # Vertical Coma
    9:  (3,  1),   # Horizontal Coma
    10: (3,  3),   # Horizontal Trefoil
    11: (4, -4),   # Oblique Tetrafoil
    12: (4, -2),   # Oblique Secondary Astigmatism
    13: (4,  0),   # Primary Spherical Aberration
    14: (4,  2),   # Vertical Secondary Astigmatism
    15: (4,  4),   # Vertical Tetrafoil
}


def incoherent_zernike_psf(N=256, defocus = None, stig = None): # zernike_coeffs=None):
    if defocus is None: # if zernike_coeffs is None:
        zernike_coeffs = {
            5:2,  # defocus
        }

    else:
        zernike_coeffs = {
            5:defocus
        }

    if stig is not None:
        zernike_coeffs[6] = stig

    x = np.linspace(- 1, 1, N)
    X, Y = np.meshgrid(x, x)
    rho = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)

    aperture = rho <= 1 #boolean array

    wavefront = np.zeros_like(rho)

    for j, coeff in zernike_coeffs.items():
        n, m = NOLL_MAP[j]
        Z = zernike(n, m, rho, theta)
        Z[~aperture] = 0
        wavefront += coeff * Z

    # Pupil function with aberration
    phase = 2 * np.pi * wavefront # this could be scaled by the wavelength.
    # suggest to not to do this in this case that adds to the confusion ==> this is not the same as a coherent point source!

    pupil = np.exp(1j * phase) 
    pupil[~aperture] = 0

    # Coherent PSF
    h = fftshift(fft2(pupil)) #The point spread function (PSF), which is given by the square of the absolute value of the 
    psf_coh = np.abs(h)**2    #Fourier transform of the wavefront, shows what the intensity image of a point source would look like.

    # Incoherent PSF: autocorrelation of coherent intensity
    psf_incoh = fftconvolve(psf_coh, psf_coh, mode='same')
    psf_incoh /= psf_incoh.sum()

    return psf_incoh


def incoh_psf_fit(ac, df, st):
    model = Model(incoherent_zernike_psf)

    nrows = ac.shape[0]
    ncols = ac.shape[1]

    pars = create_params(N = dict(value = 256, vary = False),
                         defocus = dict(value = df, vary = True, min = 0),
                         stig = dict(value = st, vary = True))

    result = model.fit(ac, params = pars) # uses the above parameters and  model to fit the data

    # plt.figure()
    # plt.imshow(ac)
    # plt.contour(result.best_fit)

    return result

In [158]:
# optimization functions

def optimum_wd(wd, kernel_size = size): # try with kernel_size 5, 7, 9, 11, and whatever calibration gives me
    """
    Returns the negative sum of the autocorrelation values for the four pixels surrounding the center pixel given pars, 
    a Parameters dictionary containing the working distance range to calculate over.
    The objective function to minimize for working distance.

    Parameter phenom: a Phenom object
    Parameter wd: a float specifying the working distance 
    Parameter kernel_size: an int specifying the size of the blurring kernel
    """

    print(wd)
    phenom.SetSemWD(wd[0])
    # phenom.SetSemWD(wd)
    # print(kernel_size)

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim, ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array
    
    acdata = filtered_autocorr(img_array, kernel_size)

    sums = (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1])
    
    return sums * (-1)


def auto_focus(wd_val_min, wd_val_max):
    """
    Sets the working distance to the value corresponding to a focused image.

    Parameter wd_val_min: the minimum working distance (float) that is considered when finding the optimal working distance
    Parameter wd_val_max: the maximum working distance (float) that is considered when finding the optimal working distance
    """

    res = gp_minimize(optimum_wd, 
                      [(wd_val_min, wd_val_max)], 
                      acq_func = "EI", 
                      n_calls = 15, 
                      n_initial_points = 5, 
                      verbose = False, 
                      random_state = 1234) 

    print(res.x[0])
    phenom.SetSemWD(res.x[0])
    return res.x[0]


def optimum_xstig(xst, kernel_size = x_size):
    print(xst)
    phenom.SetSemStigmate(ppi.Position(xst[0], 0))

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array

    acdata = filtered_autocorr(img_array, kernel_size) # calculates the filtered auto_correlation data

    sums = (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1])
    
    return sums * (-1)

    acdata = auto_correlation(img_array)

    plt.figure()
    plt.subplot(1, 3, 1)
    plt.imshow(acdata)

    if acdata.sum() != 0:
        threshold = ((acdata.max() - acdata.min()) * 0.25) + acdata.min()
        acdata[acdata < threshold] = threshold
        acdata = acdata - threshold
        if acdata.sum() != 0:
            acdata /= acdata.sum()

    plt.subplot(1, 3, 2)
    plt.imshow(acdata)

    result = incoh_psf_fit(acdata, 1, 2)

    plt.subplot(1, 3, 3)
    plt.imshow(result.best_fit)

    return result.best_values['stig']


def optimum_ystig(yst, kernel_size = y_size):
    phenom.SetSemStigmate(ppi.Position(0, yst[0]))

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array

    acdata = filtered_autocorr(img_array, kernel_size) # calculates the filtered auto_correlation data

    sums = (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1])
    
    return sums * (-1)

    acdata = auto_correlation(img_array)

    plt.figure()
    plt.subplot(1, 3, 1)
    plt.imshow(acdata)

    if acdata.sum() != 0:
        threshold = ((acdata.max() - acdata.min()) * 0.25) + acdata.min()
        acdata[acdata < threshold] = threshold
        acdata = acdata - threshold
        if acdata.sum() != 0:
            acdata /= acdata.sum()

    plt.subplot(1, 3, 2)
    plt.imshow(acdata)

    result = incoh_psf_fit(acdata, 1, 2)

    plt.subplot(1, 3, 3)
    plt.imshow(result.best_fit)

    return result.best_values['stig']


def auto_stig(xst_val_min, xst_val_max, yst_val_min, yst_val_max):
    x_res = gp_minimize(optimum_xstig, 
                        [(xst_val_min, xst_val_max)], 
                        acq_func = "EI", 
                        n_calls = 15, 
                        n_initial_points = 5, 
                        verbose = False, 
                        random_state = 1234) 

    y_res = gp_minimize(optimum_ystig, 
                        [(yst_val_min, yst_val_max)], 
                        acq_func = "EI", 
                        n_calls = 15, 
                        n_initial_points = 5, 
                        verbose = False, 
                        random_state = 1234) 
                        
    print(x_res.x[0], y_res.x[0])

    phenom.SetSemStigmate(ppi.Position(x_res.x[0], y_res.x[0]))

    return x_res.x[0], y_res.x[0]

    # return x_res.x[0], y_res.x[0]

In [1]:
# auto_focus(wd_val_min, wd_val_max)

def whole_thing2(wd_val_min, wd_val_max, xst_val_min, xst_val_max, yst_val_min, yst_val_max):
    prev = None # wd value probably?
    curr = None
    res = 1

    curr_min_wd = wd_val_min
    curr_max_wd = wd_val_max
    curr_min_xst = xst_val_min
    curr_max_xst = xst_val_max
    curr_min_yst = yst_val_min
    curr_max_yst = yst_val_max

    while prev is None or np.abs(curr - prev) >= 0.000002:
        magnification = 10000

        while prev is None or np.abs(curr - prev) >= 0.000002:
            prev = curr
            phenom.SetHFW(ppi.MagnificationToFieldWidth(magnification, 0.52))

            curr = auto_focus(curr_min_wd, curr_max_wd)

            print("curr", curr)
            
            curr_min_wd = curr - (res / 1000)
            curr_max_wd = curr + (res / 1000)

            if res >= 0.1:
                res /= 10

            magnification *= 2

        phenom.SetHFW(ppi.MagnificationToFieldWidth(160000, 0.52))
        x, y = auto_stig(curr_min_xst, curr_max_xst, curr_min_yst, curr_max_yst)

        print('stig', x, y)

        curr_min_xst = x - (res / 20)
        curr_max_xst = x + (res / 20)
        curr_min_yst = y - (res / 20)
        curr_max_yst = y + (res / 20)

        prev = None

whole_thing2(wd_val_min, wd_val_max, xst_val_min, xst_val_max, yst_val_min, yst_val_max)

NameError: name 'wd_val_min' is not defined

In [151]:
# main thing 

# auto_focus(wd_val_min, wd_val_max)

def whole_thingy(wd_val_min, wd_val_max, xst_val_min, xst_val_max, yst_val_min, yst_val_max):
    prev = None # wd value probably?
    curr = None
    res = 1

    curr_min_wd = wd_val_min
    curr_max_wd = wd_val_max
    curr_min_xst = xst_val_min
    curr_max_xst = xst_val_max
    curr_min_yst = yst_val_min
    curr_max_yst = yst_val_max

    magnification = 10000
    while prev is None or np.abs(curr - prev) >= 0.000002:
        prev = curr
        phenom.SetHFW(ppi.MagnificationToFieldWidth(magnification, 0.52))

        # if magnification < 40000:
        curr = auto_focus(curr_min_wd, curr_max_wd)

        print("curr", curr)
        
        curr_min_wd = curr - (res / 1000)
        curr_max_wd = curr + (res / 1000)

        # else:
        #     curr = auto_focus(curr_min_wd, curr_max_wd)
        #     x, y = auto_stig(curr_min_xst, curr_max_xst, curr_min_yst, curr_max_yst)

        #     print("curr", curr)
        #     print('stig', x, y)

        #     curr_min_wd = curr - (res / 1000)
        #     curr_max_wd = curr + (res / 1000)

        #     curr_min_xst = x - (res / 20)
        #     curr_max_xst = x + (res / 20)
        #     curr_min_yst = y - (res / 20)
        #     curr_max_yst = y + (res / 20)

        if res >= 0.1:
            res /= 10

        magnification *= 2

whole_thingy(wd_val_min, wd_val_max, xst_val_min, xst_val_max, yst_val_min, yst_val_max)

[0.006976636664726494]
[0.010178384411165697]
[0.008121118946772709]
[0.009713599188624012]
[0.010606697713776874]
[0.00989220095531101]
[0.009985036186143484]
[0.01002187787587837]
[0.0028773554536137653]
[0.004927016665209065]
[0.00919460316115667]
[0.012]
[0.003902186063140028]
[0.009991027521705397]
[0.005951815473973726]
0.009991027521705397
curr 0.009991027521705397
[0.009986354854650697]
[0.010626704403938538]
[0.01021525131105994]
[0.0105337473594302]
[0.010712367064460772]
[0.009802263209564514]
[0.010026846303224346]
[0.008991027521705396]
[0.00940654095770022]
[0.010050456823843878]
[0.009964274107417275]
[0.009964608777128717]
[0.00996866500684246]
[0.009926192697620625]
[0.010991027521705398]
0.009964274107417275
curr 0.009964274107417275
[0.009963806840711804]
[0.010027841795640589]
[0.009986696486352729]
[0.010018546091189755]
[0.010036408061692812]
[0.00999179977350597]
[0.009997804048890491]
[0.009989773984439978]
[0.009864274107417276]
[0.010064274107417275]
[0.009923

In [161]:
curr_min_wd = wd_val_min
curr_max_wd = wd_val_max
curr_min_xst = xst_val_min
curr_max_xst = xst_val_max
curr_min_yst = yst_val_min
curr_max_yst = yst_val_max

x, y = auto_stig(curr_min_xst, curr_max_xst, curr_min_yst, curr_max_yst)

[-0.0009345334109402414]
[0.12713537644662787]
[0.04484475787090836]
[0.10854396754496048]
[0.14426790855107496]
[-0.028304492576673806]
[-0.007709398402209383]
[-0.10368140978486501]
[-0.2]
[0.004191293305630783]
[0.2]
[-0.004441108153850848]
[-0.053597592839305014]
[-0.011980803693296627]
[-0.15115173000310403]
-0.011980803693296627 -0.025830420912269947


loc 3: flat again 

fit:

80k: 
-0.05389828588398704 0.022741233837404368
-0.04698591200495289 -0.008449061704249805
-0.04256176558505803 0.02302138869420789

160k: ** the autocorrelations are tiny for some reason
-0.010921894650427644 -0.010921894650427644
-0.010921894650427644 -0.0009345334109402414 
-0.010921894650427644 0.14893263981282956

canny:

160k:
-0.006842932710138444 0.006559823427909256 
-0.014229313694899132 0.011843096301541844 
-0.011980803693296627 -0.025830420912269947

loc 2: hill / color gradient

canny:

80k:
-0.011680560856383082 -0.00576208818182522
-0.012662128241607118 -0.0009345334109402414
-0.007223663416899173 -0.002279708315059137

160k:
-0.008827642262670826 0.010865400953506243
-0.02583042048721082 -0.0009345334109402414
-0.01349281410158426 -0.0009345334109402414
-0.010389107690292665 -0.001954776860807639

fit:

80k:
-0.008647747486823981 -0.2
0.01690083601738515 -0.2
0.021257597177136767 -0.2

160k:
0.2 -0.2

loc 1: flat ** very noticeable dark splotch after moving that might have affected measurements

canny:

80k
-0.0035383231422154715 -0.006603886064991432
-0.01754747240156876 0.0073615986698800095
-0.0032104733303996436 0.002087695223529751
0.004077176488318235 -0.0009345334109402414

120k
-0.01605251279185138 -0.005343047834632658
-0.0009345334109402414 -0.00545846313633469
-0.01092446720894244 -0.009664322149297855
-0.0139681177213129 -0.0010535147393664068

160k
-0.010497296332777545 -0.005923511628920991
-0.008165666829860474 -0.005966721187960988
-0.008423329551308784 -0.0009287589765008541
-0.005832709338229242 -0.0066829276591297004
____________________________________________________

fit:

80k
-0.0029085987130700874 -0.0033623704682875233
-0.0009345334109402414 -0.006204901711568095
-0.2 -0.0009345334109402414
-0.0009345334109402414 -0.020111266989610327

120k
-0.12673400328031975 -0.019243473962440166
-0.12079314964623254 0.0018512619088103255
-0.12891014080430768 -0.031697661906943175
-0.12689764123462877 -0.19614231863079365

160k
0.04484475787090836 0.050670661094151914
0.0396022206673233 -0.19995239224031827
0.06688424351140915 0.05226987616691969
0.04931435878042173 -0.16253988103815006